# CNN-Based Cataract Disease Prediction — Exploratory Data Analysis
**Dataset**: ODIR-5K (Ocular Disease Intelligent Recognition)  
**Task**: Binary Classification — Cataract vs Normal  
**Model**: VGG-16 Transfer Learning  
**Author**: Aiyesha Rukhsar, M.Tech CSE, NIT Delhi


## 1. Imports & Setup

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import cv2
from pathlib import Path
from tqdm import tqdm

from config import *

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_style('whitegrid')

print('Setup complete. BASE_DIR:', BASE_DIR)

## 2. Load ODIR-5K Annotations

In [ ]:
from src.preprocess import parse_odir_annotations, split_dataset, get_class_weights

df = parse_odir_annotations()
df.head()

In [ ]:
print('Dataset shape:', df.shape)
print()
print(df['class_name'].value_counts())
print()
print('Missing values:')
print(df.isnull().sum())

## 3. Class Distribution

In [ ]:
counts = df['label'].value_counts().sort_index()
labels = CLASS_NAMES
colors = ['#27ae60', '#e74c3c']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
bars = ax1.bar(labels, counts.values, color=colors, edgecolor='white', linewidth=1.5, width=0.5)
for bar, cnt in zip(bars, counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             f'{cnt}\n({cnt/len(df)*100:.1f}%)', ha='center', fontsize=11, fontweight='bold')
ax1.set_title('Class Distribution — ODIR-5K', fontsize=12, fontweight='bold')
ax1.set_ylabel('Number of images')
ax1.grid(axis='y', alpha=0.3)

# Pie chart
ax2.pie(counts.values, labels=labels, colors=colors, autopct='%1.1f%%',
        startangle=140, pctdistance=0.75,
        wedgeprops=dict(edgecolor='white', linewidth=2))
ax2.set_title('Class Proportion', fontsize=12, fontweight='bold')

plt.suptitle('ODIR-5K Binary Dataset (Cataract vs Normal)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

imbalance = counts.max() / counts.min()
print(f'\nImbalance ratio: {imbalance:.2f}x — class weighting will be applied during training.')

## 4. Sample Fundus Images

In [ ]:
normal_samples   = df[df['label'] == 0].sample(4, random_state=42)
cataract_samples = df[df['label'] == 1].sample(4, random_state=42)
samples = pd.concat([normal_samples, cataract_samples]).sample(frac=1, random_state=0)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for ax, (_, row) in zip(axes, samples.iterrows()):
    img = cv2.imread(str(row['image_path']))
    if img is not None:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, IMAGE_SIZE)
        ax.imshow(img)
    color = '#e74c3c' if row['label'] == 1 else '#27ae60'
    ax.set_title(f"{row['class_name']}\n{row['eye']} Eye", color=color,
                 fontsize=10, fontweight='bold')
    ax.axis('off')

plt.suptitle('Sample Fundus Images from ODIR-5K', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Image Dimension Analysis

In [ ]:
widths, heights, channels = [], [], []
sample_rows = df.sample(min(200, len(df)), random_state=42)

for _, row in tqdm(sample_rows.iterrows(), total=len(sample_rows), desc='Reading dims'):
    img = cv2.imread(str(row['image_path']))
    if img is not None:
        h, w, c = img.shape
        heights.append(h)
        widths.append(w)
        channels.append(c)

print(f'Width  — min:{min(widths)}  max:{max(widths)}  mean:{np.mean(widths):.0f}')
print(f'Height — min:{min(heights)} max:{max(heights)} mean:{np.mean(heights):.0f}')
print(f'\nAll images will be resized to {IMAGE_SIZE[0]}×{IMAGE_SIZE[1]} for VGG-16')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(widths,  bins=20, color='#1976d2', edgecolor='white')
axes[0].set_title('Image Width Distribution',  fontsize=11)
axes[0].axvline(224, color='red', linestyle='--', label='VGG-16 target (224)')
axes[0].legend()
axes[1].hist(heights, bins=20, color='#388e3c', edgecolor='white')
axes[1].set_title('Image Height Distribution', fontsize=11)
axes[1].axvline(224, color='red', linestyle='--', label='VGG-16 target (224)')
axes[1].legend()
plt.tight_layout()
plt.show()

## 6. Brightness & Contrast Analysis

In [ ]:
brightness = {'Normal': [], 'Cataract': []}

for _, row in tqdm(sample_rows.iterrows(), total=len(sample_rows), desc='Brightness'):
    img = cv2.imread(str(row['image_path']))
    if img is not None:
        grey = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        brightness[row['class_name']].append(grey.mean())

fig, ax = plt.subplots(figsize=(8, 4))
for cls, vals, col in zip(['Normal','Cataract'], brightness.values(), ['#27ae60','#e74c3c']):
    ax.hist(vals, bins=20, alpha=0.65, color=col, label=cls, edgecolor='white')
ax.set_title('Mean Pixel Brightness by Class', fontsize=12, fontweight='bold')
ax.set_xlabel('Mean Pixel Value (0–255)')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Data Augmentation Preview

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

aug = ImageDataGenerator(
    rotation_range=AUG_ROTATION_RANGE,
    zoom_range=AUG_ZOOM_RANGE,
    horizontal_flip=AUG_HORIZONTAL_FLIP,
    brightness_range=AUG_BRIGHTNESS_RANGE,
    shear_range=0.08,
    fill_mode=AUG_FILL_MODE
)

sample_path = df[df['label'] == 1]['image_path'].iloc[0]
orig = cv2.cvtColor(cv2.imread(sample_path), cv2.COLOR_BGR2RGB)
orig = cv2.resize(orig, IMAGE_SIZE)

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()
axes[0].imshow(orig)
axes[0].set_title('Original', fontweight='bold', color='navy')
axes[0].axis('off')

img_arr = orig.reshape(1, *IMAGE_SIZE, 3)
for i, batch in enumerate(aug.flow(img_arr, batch_size=1, seed=i), start=1):
    if i >= len(axes): break
    axes[i].imshow(batch[0].astype('uint8'))
    axes[i].set_title(f'Augmented #{i}', fontsize=9)
    axes[i].axis('off')

plt.suptitle('Data Augmentation Examples — Cataract Sample', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Train / Val / Test Split

In [ ]:
train, val, test = split_dataset(df)
class_weights = get_class_weights(train['label'].values)

split_data = {'Train': len(train), 'Val': len(val), 'Test': len(test)}
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(split_data.keys(), split_data.values(),
              color=['#1976d2','#7b1fa2','#c62828'], edgecolor='white', linewidth=1.5, width=0.5)
for bar, cnt in zip(bars, split_data.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(cnt), ha='center', fontsize=12, fontweight='bold')
ax.set_title('Dataset Split (70 / 15 / 15)', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of samples')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 9. VGG-16 Model Summary

In [ ]:
from src.model import build_vgg16_model, print_trainable_summary
model = build_vgg16_model(freeze_base=True)
print_trainable_summary(model)

In [ ]:
# Show VGG-16 layer names and output shapes
for layer in model.layers[:20]:
    print(f'  {layer.name:<28} trainable={str(layer.trainable):<6}  output={layer.output_shape}')